# SDE-Net direct multi-horizon post-hoc con label STGAN CNN (kernel 3x3)

Questo notebook **non riaddestra SDE-Net o STGAN**. Legge l'unico `predictions.csv` del modello diretto t+1,…,t+6, rimuove dall'analisi fisica i dropout solari regionali e la loro ora di recovery, ricalcola il top-5% STGAN sui soli punti validi e rigenera l'analisi normal/rare.

La decisione STGAN originale resta disponibile come `detector_is_anomaly_original`; `anomaly_group` usa invece la graduatoria pulita. I punti rimossi non diventano normali: sono esportati separatamente come anomalie di qualità. `event_group` non viene creato.

La selezione attuale legge STGAN CNN ConvGRU kernel 3x3 (`convgru_reference`, patch 3x3, seed 20); l'ablation kernel 1x1 e in `convgru_patch3_kernel1`. Ogni esecuzione completa salva una nuova analisi in una sottocartella dedicata: i risultati precedenti sono conservati.


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'physiq_pv').is_dir():
    for parent in Path.cwd().resolve().parents:
        if (parent / 'physiq_pv').is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.experiments import sde_pipeline as pipe
from physiq_pv.reporting.pointwise_detector_posthoc import build_pointwise_detector_evaluation
print('repo root:', ROOT)

## 1. Percorsi e configurazione

La cella seguente seleziona **STGAN CNN, kernel 3x3, seed 20** e la cartella di output
`outputs/sde_stgan_cnn_quality_filtered`. Sovrascrive eventuali selezioni
precedenti rimaste nell'ambiente del kernel. Per cambiare detector modifica questa
cella; per un altro kernel basta cambiare `KERNEL_SIZE` (1, 3 o 5): il nome della
run segue la stessa regola del workflow CNN.

Ogni esecuzione della configurazione crea una nuova sottocartella con data UTC
e identificativo univoco; il nome contiene anche `top5pct` per distinguere questa analisi dalle precedenti al top-1%. Riesegui dall'inizio con **Run All** per applicare
la selezione; rieseguire soltanto i grafici aggiorna l'analisi corrente.

Restano configurabili `SDE_MULTIHORIZON_PREDICTIONS`, `PVGIS_2019_FILE` e
`STGAN_MANIFEST`. Le previsioni SDE-Net e il filtro qualita restano gli stessi.


In [ ]:
# Run da analizzare: STGAN CNN (ConvGRU), seed 20.
# KERNEL_SIZE: 3 = baseline convgru_reference (corrente), 1 o 5 = ablation.
KERNEL_SIZE = 3
PATCH_SIZE = 3
# Stessa regola di stgan_cnn_pvgis_workflow.ipynb per il nome della cartella.
STGAN_RUN_NAME = ('convgru_reference' if (PATCH_SIZE, KERNEL_SIZE) == (3, 3)
    else f'convgru_patch{PATCH_SIZE}_kernel{KERNEL_SIZE}')
os.environ['STGAN_SEED_DIR'] = str(
    ROOT / 'outputs' / 'pvgis_stgan_cnn' / STGAN_RUN_NAME / 'seed_20'
)
os.environ['STGAN_POSTHOC_ROOT'] = str(
    ROOT / 'outputs' / 'sde_stgan_cnn_quality_filtered'
)
print('Run selezionata:', os.environ['STGAN_SEED_DIR'])
print('Cartella analisi:', os.environ['STGAN_POSTHOC_ROOT'])


In [ ]:
FORECAST_HORIZONS = pipe.FORECAST_HORIZONS
STGAN_SEED = 20
STGAN_CNN_OUT_DIR = Path(os.environ.get(
    'STGAN_CNN_OUT_DIR', ROOT / 'outputs' / 'pvgis_stgan_cnn' / 'convgru_reference'
)).resolve()
STGAN_SEED_DIR = Path(os.environ.get(
    'STGAN_SEED_DIR', STGAN_CNN_OUT_DIR / f'seed_{STGAN_SEED}'
)).resolve()
STGAN_SCORES = STGAN_SEED_DIR / 'anomaly_scores.csv'
PVGIS_2019 = Path(os.environ.get(
    'PVGIS_2019_FILE', ROOT / 'data' / 'pvgis' / 'piedmont_pvgis_2019.nc'
)).resolve()
STGAN_PREPARED_MANIFEST = Path(os.environ.get(
    'STGAN_MANIFEST', ROOT / 'outputs' / 'pvgis_stgan' / 'prepared' / 'manifest.csv'
)).resolve()
PVGIS_QUALITY_CANDIDATES = (
    PVGIS_2019, STGAN_PREPARED_MANIFEST,
    ROOT / 'outputs/pvgis_stgan/prepared/manifest_shard_0000.csv',
    ROOT / 'outputs/pvgis_stgan_cnn/prepared/manifest.csv',
    ROOT / 'outputs/pvgis_stgan_cnn/prepared/manifest_shard_0000.csv',
)
PVGIS_QUALITY_SOURCE = next(
    (path for path in PVGIS_QUALITY_CANDIDATES if path.is_file()),
    PVGIS_2019,
)
CLEAN_TOP_K_PERCENT = 5.0

# È lo stesso run diretto del notebook SDE-Net principale. STGAN sostituisce
# le label MTGFlow soltanto nella valutazione post-hoc.
BASE_CONFIG = {**pipe.DEFAULT_CONFIG,
    'name': 'paper_faithful_gaussian_detector_mtgflow_ep60',
    'forecast_horizons': ','.join(map(str, FORECAST_HORIZONS)),
    'epochs': 60, 'batch_size': 16, 'lr': 1e-4, 'lr_g': 1e-2,
    'dropout': 0.0, 'train_normal_only': False, 'anomaly_source': 'detector',
    'ood_smoke_test': True, 'sde_sigma_warmup_epochs': 30,
    'irradiance_loss_weight': 0.1, 'detector_regional_quantile': 0.975,
}
SDE_PREDICTIONS = Path(os.environ.get(
    'SDE_MULTIHORIZON_PREDICTIONS',
    ROOT / pipe.make_out_dir(BASE_CONFIG) / 'predictions.csv',
)).resolve()
POSTHOC_ROOT = Path(os.environ.get(
    'STGAN_POSTHOC_ROOT', ROOT / 'outputs' / 'sde_stgan_cnn_quality_filtered'
)).resolve()
# Un nuovo percorso per ogni esecuzione della configurazione, anche con root personalizzata.
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '_' + uuid4().hex[:8]
EVALUATION_DIR = POSTHOC_ROOT / f'{STGAN_SEED_DIR.parent.name}_{STGAN_SEED_DIR.name}_top{CLEAN_TOP_K_PERCENT:g}pct_{RUN_ID}'
MIN_MATCH_FRACTION = 0.90
RUN_ANALYSIS = True

print('horizons:', FORECAST_HORIZONS)
print('STGAN scores:', STGAN_SCORES)
print('Top-K sui dati quality filtered [%]:', CLEAN_TOP_K_PERCENT)
print('PVGIS quality source:', PVGIS_QUALITY_SOURCE)
print('SDE direct predictions:', SDE_PREDICTIONS)
print('evaluation output:', EVALUATION_DIR)

In [ ]:
missing = [path for path in (STGAN_SCORES, SDE_PREDICTIONS) if not path.is_file()]
if missing:
    raise FileNotFoundError('File mancanti:\n' + '\n'.join(map(str, missing)))
if not PVGIS_QUALITY_SOURCE.is_file():
    raise FileNotFoundError(
        'Nessuna sorgente PVGIS per il controllo qualità. Percorsi provati:\n'
        + '\n'.join(map(str, PVGIS_QUALITY_CANDIDATES))
    )
stgan_header = set(pd.read_csv(STGAN_SCORES, nrows=0).columns)
required_stgan = {'location', 'timestamp', 'anomaly_score', 'threshold', 'is_anomaly'}
if not required_stgan <= stgan_header:
    raise ValueError(f'Colonne STGAN mancanti: {sorted(required_stgan - stgan_header)}')
sde_header = set(pd.read_csv(SDE_PREDICTIONS, nrows=0).columns)
required_sde = {
    'location', 'timestamp', 'issue_timestamp', 'horizon_hours', 'y_true',
    'solar_irradiance_poa_target',
}
if not required_sde <= sde_header or not ({'y_pred', 'y_pred_mean'} & sde_header):
    raise ValueError(f'Schema SDE-Net diretto non compatibile: {sorted(sde_header)}')
saved_horizons = tuple(sorted(pd.read_csv(
    SDE_PREDICTIONS, usecols=['horizon_hours']
)['horizon_hours'].drop_duplicates().astype(int)))
if saved_horizons != FORECAST_HORIZONS:
    raise ValueError(f'Orizzonti salvati {saved_horizons}, attesi {FORECAST_HORIZONS}.')
print('OK: CSV diretto t+1,...,t+6, STGAN e PVGIS 2019 disponibili')

## 2. Join puntuale STGAN → canali diretti SDE-Net

Le label MTGFlow già presenti vengono rimosse. Il filtro identifica senza date hard-coded gli azzeramenti regionali isolati di direct, diffuse, sun height e produzione PV, esclude anche la recovery t+1 e ricostruisce un budget globale top-5% sulle coordinate valide. Il 5% e una scelta di questa analisi posthoc; il protocollo di riferimento del paper usa l'1%. Gli score salvati vengono riutilizzati senza riaddestramento.

In [ ]:
# Una directory gia popolata viene rifiutata prima di scrivere qualsiasi risultato.
# Per ripartire, rieseguire la configurazione: genera un nuovo EVALUATION_DIR.
RELABEL_RESULT = build_pointwise_detector_evaluation(
    SDE_PREDICTIONS, STGAN_SCORES, EVALUATION_DIR,
    min_match_fraction=MIN_MATCH_FRACTION,
    allow_overwrite=False,
    pvgis_quality_source=PVGIS_QUALITY_SOURCE,
    clean_top_k_percent=CLEAN_TOP_K_PERCENT,
)

metadata_path = EVALUATION_DIR / 'evaluation_source.json'
if not metadata_path.is_file():
    raise FileNotFoundError(metadata_path)
AUDIT = json.loads(metadata_path.read_text(encoding='utf-8'))
display(pd.DataFrame([AUDIT])[[
    'detector', 'forecast_mode', 'horizons_hours', 'source_prediction_rows',
    'detector_matched_rows_before_quality_filter', 'matched_rows',
    'excluded_unmatched_rows', 'excluded_data_quality_rows', 'match_fraction',
    'solar_dropout_timestamps', 'data_quality_timestamps', 'clean_top_k_percent',
    'eligible_detector_coordinates', 'data_quality_detector_coordinates',
    'original_detector_anomalies', 'clean_detector_anomalies',
    'normal_rows', 'rare_rows',
]])

## 3. Audit delle anomalie di qualità

Queste coordinate vengono conservate come anomalie del dato, ma non partecipano alla graduatoria top-5% fisica né alle metriche normal/rare.

In [ ]:
QUALITY_ISSUES = pd.read_csv(
    EVALUATION_DIR / 'pvgis_data_quality_issues.csv', parse_dates=['timestamp', 'source_dropout_timestamp']
)
QUALITY_PREDICTIONS = pd.read_csv(
    EVALUATION_DIR / 'data_quality_predictions.csv',
    usecols=lambda c: c in {
        'location', 'timestamp', 'horizon_hours', 'detector_is_anomaly_original'
    },
)
display(QUALITY_ISSUES)
display(QUALITY_PREDICTIONS.groupby('horizon_hours').agg(
    excluded_rows=('location', 'size'),
    originally_flagged=('detector_is_anomaly_original', 'sum'),
))

In [ ]:
joined_path = EVALUATION_DIR / 'predictions.csv'
joined = pd.read_csv(joined_path, usecols=lambda c: c in {
    'location', 'timestamp', 'horizon_hours', 'anomaly_group', 'event_group',
    'detector_is_anomaly', 'detector_is_anomaly_original', 'detector_anomaly_score',
    'solar_irradiance_poa_target'
})
assert 'anomaly_group' in joined
assert 'event_group' not in joined
assert tuple(sorted(joined['horizon_hours'].unique())) == FORECAST_HORIZONS
assert not joined.duplicated(['location', 'timestamp', 'horizon_hours']).any()
print('OK: t+1,...,t+6 usano il top-5% STGAN ricalcolato sui target validi')

In [ ]:
clean_labels = joined.loc[
    joined['horizon_hours'].eq(1) & joined['solar_irradiance_poa_target'].gt(10.0)
].copy()
clean_labels['timestamp'] = pd.to_datetime(clean_labels['timestamp'])
CLEAN_TIMESTAMP_RANKING = clean_labels.groupby('timestamp', as_index=False).agg(
    n_scored=('location', 'size'),
    n_anomalies=('detector_is_anomaly', 'sum'),
    anomaly_score_mean=('detector_anomaly_score', 'mean'),
    anomaly_score_max=('detector_anomaly_score', 'max'),
)
CLEAN_TIMESTAMP_RANKING['anomaly_share'] = (
    CLEAN_TIMESTAMP_RANKING['n_anomalies'] / CLEAN_TIMESTAMP_RANKING['n_scored']
)
CLEAN_TIMESTAMP_RANKING = CLEAN_TIMESTAMP_RANKING.sort_values(
    ['n_anomalies', 'anomaly_score_max'], ascending=False
)
CLEAN_DAILY_RANKING = (
    CLEAN_TIMESTAMP_RANKING.assign(day=lambda frame: frame['timestamp'].dt.floor('D'))
    .groupby('day', as_index=False)
    .agg(n_anomalies=('n_anomalies', 'sum'), peak_share=('anomaly_share', 'max'),
         peak_score=('anomaly_score_max', 'max'))
    .sort_values(['n_anomalies', 'peak_score'], ascending=False)
)
CLEAN_TIMESTAMP_RANKING.to_csv(EVALUATION_DIR / 'stgan_clean_daytime_timestamp_ranking.csv', index=False)
CLEAN_DAILY_RANKING.to_csv(EVALUATION_DIR / 'stgan_clean_daytime_daily_ranking.csv', index=False)
display(Markdown('### Nuovi timestamp STGAN diurni più anomali'))
display(CLEAN_TIMESTAMP_RANKING.head(20))
display(Markdown('### Nuovi giorni STGAN diurni più anomali'))
display(CLEAN_DAILY_RANKING.head(20))

## Serie temporale dell'anomaly score STGAN

Score originali sull'asse y e tempo sull'asse x, per una località selezionabile.
Le etichette e la soglia seguono il top-5% globale ricalcolato dopo il filtro
qualità, prima di selezionare località o date. A parità di score sul limite,
il rango determina l'etichetta esatta. Le ore escluse restano interruzioni.
Con località non specificata, mostriamo quella con più anomalie (scelta
esplorativa dichiarata). Nessuna media spaziale degli score viene classificata.

In [ ]:
from physiq_pv.reporting.input_target_cases import (
    load_clean_stgan_labels, plot_stgan_score_timeline,
)

SCORE_LOCATION = os.environ.get('STGAN_SCORE_LOCATION') or None
SCORE_START = os.environ.get('STGAN_SCORE_START') or None  # ISO timestamp UTC
SCORE_END = os.environ.get('STGAN_SCORE_END') or None
score_labels = load_clean_stgan_labels(
    STGAN_SCORES, excluded_timestamps=QUALITY_ISSUES['timestamp'],
    top_percent=CLEAN_TOP_K_PERCENT,
)
score_figure, score_series = plot_stgan_score_timeline(
    score_labels, EVALUATION_DIR / 'score_timeline',
    location=SCORE_LOCATION, start=SCORE_START, end=SCORE_END,
)
print('Località:', score_series['location'].iloc[0])
print('Soglia score globale:', score_labels.attrs['score_cutoff'])
print('Ore visualizzate:', len(score_series))
display(Image(filename=str(score_figure)))

## Serie temporale regionale STGAN — tutte le località

La curva superiore mostra, per ogni ora, la **percentuale di località anomale
fra quelle con score valido**. La heatmap inferiore conserva tutte le località:
una riga per località, una colonna per giorno e colore pari alla percentuale
di ore valide del giorno classificate anomale. Grigio = nessuna ora valida.

Entrambi i pannelli riusano le etichette del top-5% globale pulito, già
calcolate prima di selezionare le date. Si includono tutte le ore con score,
anche notturne. Le osservazioni mancanti o escluse per qualità non diventano
normali. Non si applica la soglia dello score alla curva delle percentuali.
La scala della heatmap è fissa 0–100%; le righe sono ordinate per ID, senza
selezionare soltanto le località più anomale. I CSV esportano anche i
denominatori, per riconoscere ore o giorni con copertura ridotta.

`REGIONAL_START` e `REGIONAL_END` permettono di ingrandire un intervallo
(timestamp ISO UTC, estremi inclusi). Con `None` viene mostrato tutto il
periodo disponibile. La selezione della località nel grafico precedente
non influenza questa figura.

In [ ]:
from physiq_pv.reporting.stgan_regional import build_stgan_regional_overview
from physiq_pv.reporting.input_target_cases import load_clean_stgan_labels

REGIONAL_START = os.environ.get('STGAN_REGIONAL_START') or None
REGIONAL_END = os.environ.get('STGAN_REGIONAL_END') or None
# Riuso le label della cella precedente; posso eseguire anche questa cella da sola.
if 'score_labels' not in globals():
    score_labels = load_clean_stgan_labels(
        STGAN_SCORES, excluded_timestamps=QUALITY_ISSUES['timestamp'],
        top_percent=CLEAN_TOP_K_PERCENT,
    )
REGIONAL_PATHS = build_stgan_regional_overview(
    score_labels, EVALUATION_DIR / 'score_timeline',
    start=REGIONAL_START, end=REGIONAL_END,
)
display(Image(filename=str(REGIONAL_PATHS['figure'])))
regional_hourly = pd.read_csv(REGIONAL_PATHS['hourly'])
display(Markdown('### Ore con la maggiore percentuale di località anomale'))
display(regional_hourly.nlargest(10, 'anomaly_share_pct'))
print('Figura e tabelle regionali:', {key: str(path) for key, path in REGIONAL_PATHS.items()})
del score_labels

## Errori per tipo di estremo meteorologico (quality filtered)

Sette gruppi: **normali STGAN**, **temperatura estrema alta**, **temperatura
estrema bassa**, **vento estremo forte**, **vento estremo debole**,
**irradianza estrema alta**, **irradianza estrema bassa**. I sei gruppi
estremi sono l'intersezione fra il top-5% STGAN pulito e le etichette
climatologiche esistenti, unite sulla stessa localita e sul timestamp target.
STGAN non fornisce un'attribuzione causale per variabile. I normali mantengono
la decisione STGAN; le anomalie senza etichette meteo restano fuori dai sei
gruppi estremi e sono contate come `untyped_anomaly`. Le condizioni concomitanti
compaiono in ciascun gruppo pertinente (`multiple_conditions` nell'audit).

Il CSV climatologico predefinito usa il riferimento 2005-2018, finestra
stagionale +/-15 giorni e quantili 2.5%-97.5%; `PVGIS_CLIMATOLOGY_SCORES`
permette di selezionare un altro CSV. Si usa esplicitamente la variabile
`solar_irradiance_poa` per l'irradianza, senza confonderla con la produzione PV.
Le etichette di temperatura e vento sono a due code: il lato si ricava dalla
stessa riga climatologica (`value > climatology_q_high` = alta/forte,
`value < climatology_q_low` = bassa/debole). Alta e bassa sono quindi relative
alla climatologia di quel giorno dell'anno e ora, non soglie assolute.
Nelle curve RMSE i lati bassi/deboli sono tratteggiati.

Si leggono **solo le previsioni quality filtered** della presente esecuzione:
il filtro e l'audit dei dropout e delle recovery sono obbligatori e ricontrollati.
Poi si selezionano i target diurni con irradianza >10 W/m2. Gli zeri notturni
regolari non sono anomalie di qualita.

I boxplot t+1 e t+6 mostrano gli errori assoluti di tutti i campioni:
rombo nero = **MAE**, box = quartili, baffi = Tukey; gli outlier sono nascosti
ma inclusi nelle statistiche. Le curve RMSE t+1,...,t+6 mostrano la mediana
degli RMSE calcolati per localita e le bande il 25-75 percentile fra localita,
**non un intervallo di confidenza**. Con una sola localita la banda e omessa;
con nessun campione il box e vuoto e la curva interrotta.

Ogni figura include il totale diurno e le fasce di produzione 0-20,...,80-100%,
con un unico picco di riferimento q99 dei target positivi diurni validi
(senza duplicarli per orizzonte); i valori oltre q99 confluiscono nell'ultima
fascia. I CSV riportano anche MAE e RMSE aggregati sui campioni, numerosita,
RMSE per localita e provenienza. Questa sezione precede il confronto
normal/rare generale, conservato nelle celle successive.


In [ ]:
import importlib
import physiq_pv.reporting.stgan_meteo_errors as meteo_module

# Ricarica sempre il modulo: un kernel avviato prima dell'aggiornamento
# conserverebbe la versione a cinque gruppi anche con Run All.
meteo_module = importlib.reload(meteo_module)
print('Modulo meteo:', meteo_module.__file__)
if not {'temperature_high', 'temperature_low', 'wind_high', 'wind_low'} <= set(meteo_module.CATEGORIES):
    raise RuntimeError(
        f'Modulo meteo non aggiornato ({meteo_module.__file__}): mancano i gruppi alta/bassa. '
        'Aggiornare il repository o reinstallare physiq_pv in modalita editable.'
    )
build_stgan_meteo_errors = meteo_module.build_stgan_meteo_errors

CLIMATOLOGY_SCORES = Path(os.environ.get(
    'PVGIS_CLIMATOLOGY_SCORES', ROOT / pipe.TEST_ANOMALY_SCORES,
)).resolve()
if not CLIMATOLOGY_SCORES.is_file():
    raise FileNotFoundError(
        f'Manca il CSV delle etichette climatologiche: {CLIMATOLOGY_SCORES}. '
        'Impostare PVGIS_CLIMATOLOGY_SCORES; non usare gli score STGAN come etichette meteo.'
    )
METEO_ERROR_PATHS = build_stgan_meteo_errors(
    EVALUATION_DIR, CLIMATOLOGY_SCORES, detail_horizons=(1, 6),
)
display(pd.read_csv(METEO_ERROR_PATHS['counts']))
meteo_metrics = pd.read_csv(METEO_ERROR_PATHS['metrics'])
display(meteo_metrics.loc[meteo_metrics['bin'].eq('all_daytime'), [
    'horizon_hours', 'category', 'count', 'n_locations', 'mae', 'rmse',
    'rmse_site_q1', 'rmse_site_median', 'rmse_site_q3',
]])
for key in ('mae_boxplots_t1', 'mae_boxplots_t6', 'rmse_bands'):
    display(Image(filename=str(METEO_ERROR_PATHS[key])))
print('Figure, metriche e audit:', {key: str(path) for key, path in METEO_ERROR_PATHS.items()})


## 4. Analisi post-hoc e suite per bin separate per t+1 e t+6

In [ ]:
DETAILED_HORIZONS = (1, 6)
REQUIRED_FIGURE_PREFIXES = ('mae_', 'rmse_', 'nmpil_', 'picp_', 'clc_')
FULL_POSTHOC_FIGURES = {}
for horizon_hours in DETAILED_HORIZONS:
    detail_out = EVALUATION_DIR / 'posthoc_by_horizon' / f't_plus_{horizon_hours}'
    analysis_command = pipe.build_analysis_command(
        str(detail_out), BASE_CONFIG, predictions=str(joined_path),
        horizon_hours=horizon_hours,
    )
    if RUN_ANALYSIS:
        subprocess.run(analysis_command, check=True, cwd=ROOT)
    elif not (detail_out / 'daytime_bin_anomaly_metrics.csv').is_file():
        raise FileNotFoundError(
            f'RUN_ANALYSIS=False ma manca il report per t+{horizon_hours}: {detail_out}'
        )
    figures = pipe.build_posthoc_figures(
        str(detail_out), horizon_hours=horizon_hours
    )
    missing = [
        prefix for prefix in REQUIRED_FIGURE_PREFIXES
        if not any(name.startswith(prefix) for name in figures)
    ]
    if missing:
        raise RuntimeError(
            f'Suite post-hoc STGAN incompleta per t+{horizon_hours}: {missing}'
        )
    FULL_POSTHOC_FIGURES[horizon_hours] = figures
    print(
        f't+{horizon_hours}: {len(figures)} figure STGAN per bin in {detail_out}'
    )

In [ ]:
POSTHOC_PATHS = pipe.build_direct_multihorizon_posthoc(EVALUATION_DIR)
DIRECT_METRICS = pd.read_csv(POSTHOC_PATHS['metrics'])
display(DIRECT_METRICS)
print({name: str(path) for name, path in POSTHOC_PATHS.items()})

display(Markdown('## Confronto multi-orizzonte STGAN'))
for key in ('error_figure', 'boxplot_figure', 'histogram_figure', 'prediction_figure'):
    display(Image(filename=str(POSTHOC_PATHS[key])))
for horizon_hours, figures in FULL_POSTHOC_FIGURES.items():
    display(Markdown(f'## Suite STGAN per bin — t+{horizon_hours}'))
    for name, path in sorted(figures.items()):
        display(Markdown(f'**{name}**'))
        display(Image(filename=str(path)))

## Interpretazione

Le curve confrontano MAE e RMSE sui punti validi che il **top-5% STGAN pulito** classifica normali o anomali per la stessa località e lo stesso timestamp target. Le decisioni originali e i dropout restano negli output di audit. Le suite dettagliate per bin sono calcolate separatamente per t+1 e t+6 e includono MAE, RMSE, NMPIL, PICP e CLC. Questa è un'analisi post-hoc esplorativa: non modifica il training e non interpreta più i dropout come eventi meteorologici.